# MLE_case

本笔记通过 3 个完整案例，展示最大似然估计在连续型、二元型和计数型因变量中的应用。整个分析都围绕同一条主线展开：

$$
\text{数据} \rightarrow \text{分布假设} \rightarrow \text{参数化} \rightarrow \text{对数似然} \rightarrow \text{估计与解释}
$$

数据全部来自 `MLE_codes.ipynb`，不在本笔记中重复生成。


## 0. 环境设置

先导入案例分析所需的库，并定义一个简单的对比表函数。后续三个案例共用这套环境。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy import stats

plt.rcParams.update({
    'font.family': 'Noto Sans CJK JP',
    'axes.unicode_minus': False,
    'figure.dpi': 150,
    'figure.figsize': (6, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
})


def compare_table(true_vals, ols_vals, mle_vals, names):
    return pd.DataFrame({'参数名': names, '真实值': true_vals, 'OLS 估计': ols_vals, 'MLE 估计': mle_vals})


## Case 1：正态线性模型——MLE vs. OLS

**背景**：使用模拟数据 `method_MLE_data01_normal_linear.csv`。在正态误差假设下，MLE 和 OLS 给出数值上高度一致的参数估计。  
**分析目标**：对比 OLS 与手动构造的 MLE，观察系数估计与对数似然值。


In [ ]:
df1 = pd.read_csv('./data/method_MLE_data01_normal_linear.csv')
print(df1.describe().round(3))

X1 = sm.add_constant(df1[['x1', 'x2']])
y1 = df1['y']
ols_res = sm.OLS(y1, X1).fit()
print(ols_res.summary())


def neg_loglik_normal(params, y, X):
    beta = params[:-1]
    log_sigma = params[-1]
    sigma = np.exp(log_sigma)
    resid = y - X @ beta
    n = len(y)
    return (n / 2) * np.log(2 * np.pi * sigma**2) + np.sum(resid**2) / (2 * sigma**2)

init_params = np.array([0.0, 0.0, 0.0, 0.0])
res_mle = minimize(neg_loglik_normal, init_params, args=(y1.values, X1.values), method='BFGS')
beta_mle = res_mle.x[:-1]
sigma_mle = np.exp(res_mle.x[-1])
loglik_mle = -res_mle.fun

comp1 = compare_table([1.0, 2.0, -0.5], ols_res.params.values.tolist(), beta_mle.tolist(), ['const', 'x1', 'x2'])
print(comp1.round(4))
print(f'MLE 估计得到的 sigma = {sigma_mle:.4f}')
print(f'MLE 的对数似然值 = {loglik_mle:.4f}')


从结果可以看到，OLS 与手工实现的 MLE 在系数估计上数值高度一致。这并不偶然，因为在正态同方差假设下，极大化对数似然等价于最小化残差平方和。也正因为如此，线性回归是理解 MLE 最自然的入口：它把抽象的似然思想落回到已经熟悉的 OLS 框架中。

另一个值得注意的点是：MLE 不仅给出参数估计，还天然给出对数似然值。这个量本身不是全部，但它是后续 LR 检验、AIC、BIC 等工具的共同基础。


::: {.callout-note collapse="true"}
### Stata 对应代码

```stata
* Case 1：正态线性模型
import delimited "./data/method_MLE_data01_normal_linear.csv", clear
regress y x1 x2
scalar list e(ll)
```

> **说明**：在正态同方差假设下，`regress` 的系数估计与 MLE 一致。
:::


## Case 2：Logit 模型——二元信用违约预测

**背景**：使用模拟数据 `method_MLE_data02_logit.csv`，模拟银行信用评分场景。  
**研究问题**：客户收入和年龄如何影响信用卡违约概率？


In [ ]:
df2 = pd.read_csv('./data/method_MLE_data02_logit.csv')
print(df2.describe().round(3))
print(f'整体违约率 = {df2["default"].mean():.3f}')

df2['income_group'] = pd.qcut(df2['income'], q=5, duplicates='drop')
group_rate = df2.groupby('income_group', observed=False)['default'].mean()
fig, ax = plt.subplots(figsize=(6.5, 4))
group_rate.plot(kind='bar', color='#2C6BAC', ax=ax)
ax.set_ylabel('平均违约率')
ax.set_xlabel('income 分组（五分位）')
ax.set_title('不同收入组的平均违约率')
plt.xticks(rotation=25)
plt.show()

X2 = sm.add_constant(df2[['income', 'age_std']])
y2 = df2['default']
logit_res = sm.Logit(y2, X2).fit(disp=False)
print(logit_res.summary())

margeff_logit = logit_res.get_margeff(at='overall')
print(margeff_logit.summary())

income_grid = np.linspace(2, 8, 100)
X_pred = pd.DataFrame({'const': 1.0, 'income': income_grid, 'age_std': 0.0})
p_hat = logit_res.predict(X_pred)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(income_grid, p_hat, color='#2C6BAC', lw=2)
ax.set_xlabel('income')
ax.set_ylabel('预测违约概率')
ax.set_title('固定 age_std = 0 时的预测概率曲线')
plt.show()

logit_null = sm.Logit(y2, np.ones((len(y2), 1))).fit(disp=False)
comp2 = pd.DataFrame({'模型': ['零模型（仅截距）', '完整模型（income + age_std）'],
                      'Log-Likelihood': [logit_null.llf, logit_res.llf],
                      'AIC': [logit_null.aic, logit_res.aic]})
print(comp2.round(4))


这个案例里，最重要的不是“系数显著不显著”本身，而是要看懂软件输出里每个统计量在 MLE 框架中的位置。`Log-Likelihood` 是收敛后对数似然的取值；`LLR p-value` 用于检验解释变量整体是否带来显著改进；`Pseudo R-squ.` 则是基于零模型和完整模型的对数似然构造出来的辅助指标。

另一个关键点是：Logit 模型中的系数不能像 OLS 那样直接解释为“$x$ 增加 1 单位，$y$ 增加多少”。这里更适合结合平均边际效应和预测概率曲线来读。


::: {.callout-note collapse="true"}
### Stata 对应代码

```stata
* Case 2：Logit 模型
import delimited "./data/method_MLE_data02_logit.csv", clear
logit default income age_std
margins, dydx(*)
predict p_hat, pr
margins, at(income=(2(0.5)8) age_std=0)
marginsplot, noci
```
:::


## Case 3：Poisson 模型——月度交易计数

**背景**：使用模拟数据 `method_MLE_data03_poisson.csv`，模拟客户月度交易次数。  
**研究问题**：收入和从业年限如何影响交易次数的条件期望？


In [ ]:
df3 = pd.read_csv('./data/method_MLE_data03_poisson.csv')
print(df3.describe().round(3))
mean_count = df3['trade_count'].mean()
std_count = df3['trade_count'].std(ddof=1)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(df3['trade_count'], bins=np.arange(-0.5, df3['trade_count'].max() + 1.5, 1),
        density=True, alpha=0.65, color='#2C6BAC', label='计数数据直方图')
counts = np.arange(df3['trade_count'].max() + 1)
ax.plot(counts, stats.norm.pdf(counts, loc=mean_count, scale=std_count), color='#B8C500', lw=2, label='同均值正态密度')
ax.set_xlabel('月度交易次数')
ax.set_ylabel('密度 / 概率')
ax.set_title('计数数据与正态分布对比')
ax.legend(frameon=False)
plt.show()

X3 = sm.add_constant(df3[['income', 'experience']])
y3 = df3['trade_count']
pois_res = sm.Poisson(y3, X3).fit(disp=False)
print(pois_res.summary())
print(pois_res.get_margeff(at='overall').summary())

income_grid = np.linspace(-2, 2, 100)
X_pred3 = pd.DataFrame({'const': 1.0, 'income': income_grid, 'experience': 0.0})
yhat_count = pois_res.predict(X_pred3)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(income_grid, yhat_count, color='#2C6BAC', lw=2)
ax.set_xlabel('income')
ax.set_ylabel('预测交易次数')
ax.set_title('固定 experience = 0 时的预测交易次数曲线')
plt.show()


Poisson 案例最值得注意的地方，在于参数解释方式与线性回归明显不同。Poisson 使用对数连接函数，因此系数首先作用在对数条件均值上；若进一步取指数，$\exp(\hateta_j)$ 才能解释为倍率变化。也正因为如此，Poisson 回归通常会配合边际效应或 `IRR` 一起解读。

图形对比同样重要。计数数据天然是离散、非负且常常右偏的；如果硬套正态分布，不仅形状不贴合，还会把不少概率质量放到负值区间。


::: {.callout-note collapse="true"}
### Stata 对应代码

```stata
* Case 3：Poisson 模型
import delimited "./data/method_MLE_data03_poisson.csv", clear
poisson trade_count income experience
poisson trade_count income experience, irr
margins, dydx(*)
margins, at(income=(-2(0.25)2) experience=0)
marginsplot, noci
```
:::


## 综合小结

最后，把 3 个案例放回同一张表中比较。表中的对数似然与 AIC 都直接来自各模型的估计输出，而“核心发现”则提炼每个案例最值得记住的一句话。


In [ ]:
summary_tbl = pd.DataFrame({
    '案例': ['Case 1', 'Case 2', 'Case 3'],
    '模型': ['正态线性', 'Logit', 'Poisson'],
    '因变量类型': ['连续', '二元 0/1', '非负计数'],
    '分布假设': ['正态', 'Bernoulli', 'Poisson'],
    '对数似然': [ols_res.llf, logit_res.llf, pois_res.llf],
    'AIC': [ols_res.aic, logit_res.aic, pois_res.aic],
    '核心发现': ['OLS 系数 = MLE 系数', '收入影响违约概率', '收入提升交易期望频率']
})
print(summary_tbl.round(4))


三个案例，三种不同的因变量类型，三种不同的分布假设，对应三种不同的似然函数。但它们背后的估计逻辑是完全统一的：先根据因变量性质选择分布，再把分布参数写成解释变量的函数，最后交给软件极大化对数似然。这正是 MLE 作为“统一估计语言”的价值所在。
